In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

def find_repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts" / "generate_finer_cam_panderm.py").exists():
            return p.resolve()
    raise FileNotFoundError


REPO = find_repo_root()
for extra in [REPO, REPO / "external" / "PanDerm" / "classification"]:
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from src.thesis.config import *
from src.thesis import data, models, cams, metrics, plotting, audit

EV = data.load_eval()
LESION = data.load_lesions(EV)
QC = data.load_qc()
HUM = data.load_hum(QC, LESION)
X_TEST, Y_MEL, ID_TEST, TEST_DF = data.load_test()
M0, M5 = models.ck5("ha0"), models.ck5("ha5")

AUD = audit.run_audit(EV, LESION, HUM, X_TEST, Y_MEL, ID_TEST, QC, M0, M5)

                                       check  pass         detail
                            index convention  True               
                  test split 987 with 54 MEL  True       987 / 54
                  no annotated image in test  True               
                 lesion masks non degenerate  True             34
             HUM has positives and negatives  True             31
                          ha0 and ha5 differ  True               
                 target follows ground truth  True               
            FinerCAM alpha 0 equals Grad-CAM  True       1.000000
                          AUC rank invariant  True               
                          AUC argument order  True               
              HUM identical to 14d reference  True 31 ref, 31 new
            centre baseline reproduces 0.828  True          0.828
aligned target vs clinician reproduces 0.743  True          0.743

all 13 checks pass


In [2]:
import pandas as pd
import numpy as np

# ---- splits, identical across folds except the held out annotated images ----
d0 = pd.read_csv(MELNV / "f2_fold0.csv")
split_tab = (d0.groupby(["split", "gt_label"]).size().unstack(fill_value=0)
             .reindex(["train", "val", "test"]))
split_tab["total"] = split_tab.sum(axis=1)
split_tab["MEL share"] = (split_tab["MEL"] / split_tab["total"]).round(3)
print("SPLITS, fold 0\n", split_tab.to_string(), "\n")

# ---- per fold composition of the concept supervision ------------------------
FM = data.load_feat_manifest()
ANN = FM.query("rule != 'nevus_anchor'")
fold_rows = []
for k in range(5):
    d = pd.read_csv(MELNV / f"f2_fold{k}.csv")
    h = pd.read_csv(MELNV / f"f2_fold{k}_heldout.csv")
    sup = d[(d.split == "train") & (d.has_feat == 1)]
    fold_rows.append({"fold": k, "train": int((d.split == "train").sum()),
                      "HA2 supervised": len(sup),
                      "HA2 MEL dx": int((sup.gt_label == "MEL").sum()),
                      "HA2 NV dx": int((sup.gt_label == "NV").sum()),
                      "held out": len(h), "test": int((d.split == "test").sum())})
fold_tab = pd.DataFrame(fold_rows).set_index("fold")
print("FOLDS\n", fold_tab.to_string(), "\n")

# ---- HA2 rules, annotated images only ---------------------------------------
rule_tab = ANN.groupby(["rule", "dx"]).size().unstack(fill_value=0)
print("HA2 RULES, 32 annotated images\n", rule_tab.to_string(), "\n")

# ---- annotations: labels, groups, coverage ----------------------------------
ann_img = QC.groupby("Image_ID").agg(n_ann=("label", "size"),
                                     n_labels=("label", "nunique"))
cov = pd.Series({i: HUM[i][LESION[i]].mean() for i in HUM}, name="MEL union share of lesion")
les = pd.Series({i: int(LESION[i].sum()) for i in HUM}, name="lesion patches")
ann_sum = pd.DataFrame({
    "annotations per image": metrics.summarise(ann_img.n_ann, 1),
    "distinct labels per image": metrics.summarise(ann_img.n_labels, 1),
    "lesion patches": metrics.summarise(les, 1),
    "MEL union share of lesion": metrics.summarise(cov, 2),
}).T
print("ANNOTATIONS\n", ann_sum.to_string(), "\n")

grp_tab = (QC.assign(group=QC.label.map(LAB2GRP))
           .groupby("group").agg(annotations=("label", "size"),
                                 images=("Image_ID", "nunique"))
           .sort_values("images", ascending=False))
print("MORPHOLOGY GROUPS\n", grp_tab.to_string())

# ---- write --------------------------------------------------------------------
for name, df in [("data_splits", split_tab), ("data_folds", fold_tab),
                 ("data_rules", rule_tab), ("data_annotations", ann_sum),
                 ("data_groups", grp_tab)]:
    df.to_csv(TAB / f"{name}.csv")
    df.to_latex(TAB / f"{name}.tex", escape=True)
print(f"\nwritten to {TAB}")

SPLITS, fold 0
 gt_label   MEL    NV  total  MEL share
split                                 
train     1033  5317   6350      0.163
val         22   450    472      0.047
test        54   933    987      0.055 

FOLDS
       train  HA2 supervised  HA2 MEL dx  HA2 NV dx  held out  test
fold                                                              
0      6350              25          12         13         7   987
1      6350              25          12         13         7   987
2      6351              26          12         14         6   987
3      6351              26          12         14         6   987
4      6351              26          12         14         6   987 

HA2 RULES, 32 annotated images
 dx            mel  nv
rule                 
mel_on_nevus    0  10
mel_only       14   2
nv_only         1   5 

ANNOTATIONS
                                mean_sd         median_iqr   n
annotations per image        2.9 ± 1.0     3.0 [2.0, 4.0]  34
distinct labels per image   